# 📗 OpenAI API 활용 — 텍스트 데이터 정형화

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

앞 시간에 **구조화된 출력**으로 리뷰 한 건의 감정을 `{'sentiment': ..., 'summary': ...}` 로 받았습니다. 이번 시간엔 그 기술을 **데이터 전처리 도구**로 씁니다 — 사람이 자유롭게 쓴 문장 더미를 **바로 집계·검색·저장할 수 있는 표**로 바꿉니다.

지금까지의 전처리(11일차)는 **규칙**으로 했습니다(정규표현식·형태소·불용어). 규칙은 빠르고 싸지만 "이 문의가 얼마나 급한가", "이 문장에서 개인정보는 어디까지인가" 같은 **판단**은 못 합니다. 오늘은 그 판단이 필요한 자리에 LLM 을 끼워 넣습니다.

## ⏪ 복습 — 지난 시간

- `response_format` 으로 답을 **정해진 JSON 형식**으로 받았습니다(딕셔너리 스키마 / pydantic 두 방법).
- `Literal[...]` 로 **값 후보를 못박아** 집계가 가능해진다는 것을 봤습니다(측면 이름이 흩어지던 문제).
- 오늘은 같은 도구로 **문의·개인정보·설문·대량 리뷰**를 표로 만듭니다.

**오늘의 목표**

- [ ] 자유 텍스트를 **스키마에 맞춘 표**로 바꾸고 그대로 집계한다.
- [ ] **중첩 스키마**(리스트 안에 객체)로 개인정보를 탐지하고 **마스킹**한다.
- [ ] 제각각인 설문 응답을 **표준화**하고 **품질 등급**을 매긴다.
- [ ] 수십 건을 **한 번에 처리하는 파이프라인**을 만들어 CSV 로 저장한다.
- [ ] LLM 전처리가 **규칙 기반 전처리와 무엇이 다른지** 설명한다.

아래 준비 셀들을 먼저 실행하세요(키가 없어도 저장된 응답으로 진행됩니다).

In [ ]:
# [제공 코드] OpenAI 클라이언트 준비 — 이 셀은 실행만 하세요.
# .env 에 OPENAI_API_KEY 가 있으면 실제 OpenAI 에 연결하고,
# 없으면 미리 저장해 둔 응답(data/api_cache.json)으로 진행됩니다(키·인터넷 없이 실습 가능).
# 아래 client 사용법은 공식 문서와 똑같습니다 → https://developers.openai.com/api/docs/guides/text
import sys
sys.path.insert(0, '.')          # openai_client.py 가 있는 폴더

from openai_client import get_client

client = get_client()

In [ ]:
# [제공 코드] 라이브러리
import json
import pandas as pd
from pydantic import BaseModel, Field
from typing import Literal, Optional

---
# 1. 왜 '정형화'인가 — 문장 더미로는 아무것도 못 센다

## 왜 필요할까요?
고객센터에 이런 문의가 하루 300건 들어온다고 해 봅시다.

```
"주문번호 ORDER-230415입니다. 환불 어떻게 하나요??? 급합니다ㅠㅠ 배송비 차감되나요??"
```

이 상태로는 **"환불 문의가 몇 건인가", "급한 건 몇 건인가"** 를 셀 수 없습니다. 세려면 각 문장을 **같은 칸**을 가진 행으로 바꿔야 합니다.

| 원문 | 유형 | 긴급도 | 감정 | 주문번호 |
|---|---|---|---|---|
| 주문번호 ORDER-230415입니다… | `refund` | `urgent` | `negative` | ORDER-230415 |

이 변환이 **정형화(structuring)** 입니다. 정형화가 끝나면 그 뒤는 우리가 이미 아는 pandas 세계입니다 — `value_counts()`·`groupby()`·필터·저장.

<img src="images/정형화_문장더미_vs_표.jpg" width="820">

*왼쪽처럼 크기도 모양도 제각각인 문장 더미로는 셀 수가 없습니다. 오른쪽처럼 **같은 칸**을 갖게 만드는 것이 정형화입니다.*

## 스키마 설계 3원칙
| 원칙 | 왜 | 예 |
|---|---|---|
| **셀 값은 `Literal` 로 못박는다** | 자유 문자열이면 같은 뜻이 다른 이름으로 흩어져 집계가 깨진다 | `Literal['refund','delivery',…]` |
| **없을 수 있는 값은 `Optional`** | 모든 문의에 주문번호가 있진 않다. 억지로 채우면 **없는 값을 지어낸다** | `Optional[str] = None` |
| **필드마다 `description`** | 모델이 읽는 **작성 지침**이다. 여기서 품질의 절반이 갈린다 | `Field(description='ORDER-XXXXXX 형식, 없으면 null')` |

> 세 번째가 특히 중요합니다. `description` 은 주석이 아니라 **모델에게 전달되는 문서**입니다 — 스키마를 잘 적는 것이 프롬프트를 잘 쓰는 것과 같은 일이 됩니다.

---
# 2. 고객 문의를 표로 — 분류·긴급도·감정 한 번에

## 왜 필요할까요?
문의를 사람이 하나씩 읽어 분류하면 300건에 몇 시간이 듭니다. 게다가 사람마다 기준이 흔들립니다. **스키마 하나**를 잘 정해 두면 기준이 고정되고, 새 문의가 들어올 때마다 같은 잣대로 분류됩니다.

## 이번 스키마
- `category` — 무엇에 대한 문의인가(환불·교환·배송·상품·기술·불만·기타)
- `urgency` — 얼마나 급한가(4단계)
- `emotion` — 고객이 어떤 상태인가(긍정·중립·부정)
- `key_issues` — 핵심 이슈 키워드 2~5개(리스트)
- `order_number` — 있으면 뽑고 **없으면 `None`**
- `suggested_response` — 상담원이 먼저 안내할 내용 한두 문장

In [ ]:
class Inquiry(BaseModel):
    """고객 문의 한 건을 분석한 결과."""
    category: Literal['환불', '교환', '배송', '상품', '기술지원', '불만', '기타'] = Field(
        description='문의의 주된 목적')
    urgency: Literal['긴급', '높음', '보통', '낮음'] = Field(
        description='긴급: 즉시 처리 요구 / 높음: 시간 압박 표현 / 보통: 일반 문의 / 낮음: 단순 질문')
    emotion: Literal['긍정', '중립', '부정'] = Field(description='고객의 감정 상태')
    key_issues: list[str] = Field(description='핵심 이슈를 명사형 키워드로 2~5개')
    order_number: Optional[str] = Field(default=None,
        description='ORDER-XXXXXX 형태의 주문번호. 문의에 없으면 반드시 null')
    suggested_response: str = Field(description='상담원이 먼저 안내할 내용 한두 문장')

print('스키마 필드:', list(Inquiry.model_fields))

In [ ]:
# [제공 코드] 실제 고객센터에 들어올 법한 문의 6건 (오탈자·이모티콘 그대로)
inquiries = [
    '주문번호 ORDER-230415입니다. 환불 어떻게 하나요??? 급합니다ㅠㅠ 배송비 차감되나요??',
    '배송 언제 오나요...... 일주일 지났는데 추적도 안돼요ㅠㅠ 빨리 답변 부탁드립니다',
    '상품 불량입니다. 교환 가능한가요? 포장 찢어져서 왔고 제품에 스크래치 있어요.',
    '로그인이 안 됩니다... 비밀번호 재설정 눌러도 이메일 안 와요 도와주세요ㅠ',
    '감사합니다! 빠른 배송 덕분에 생일 선물 제때 받았어요 ^^ 포장도 깔끔하네요',
    '주문번호 ORDER-230512 / 노트북 파우치 색상 변경 가능할까요? 블랙 → 네이비로요',
]
print(len(inquiries), '건')

In [ ]:
def analyze_inquiry(text):
    """문의 한 건을 Inquiry 스키마로 분석해 객체로 돌려준다."""
    resp = client.chat.completions.parse(model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': (
                '너는 고객센터 데이터 분석가야. 문의를 스키마에 맞춰 분석해.\n'
                '유형 기준 — 환불: 돈을 돌려받는 요청 / 교환: 같은 상품으로 바꾸는 요청 / '
                '배송: 배송 조회·지연·분실 / 상품: 상품 정보·재고·성분 문의 / '
                '기술지원: 로그인·결제·앱·시스템 오류 / 불만: 서비스 자체에 대한 항의 / '
                '기타: 위 어디에도 안 맞을 때만.'
            )},
            {'role': 'user', 'content': text}],
        response_format=Inquiry)
    return resp.choices[0].message.parsed

one = analyze_inquiry(inquiries[0])
print('유형:', one.category, '| 긴급도:', one.urgency, '| 감정:', one.emotion)
print('주문번호:', one.order_number)
print('이슈:', one.key_issues)
print('응답 가이드:', one.suggested_response)

여섯 건을 모두 돌려 **표**로 만들면, 그 순간부터 pandas 로 셀 수 있습니다.

In [ ]:
rows = []
for text in inquiries:
    r = analyze_inquiry(text)
    rows.append({'원문': text[:20] + '…', 'category': r.category, 'urgency': r.urgency,
                 'emotion': r.emotion, 'order_number': r.order_number})

inquiry_df = pd.DataFrame(rows)
display(inquiry_df)
print('[유형별 건수]')
display(inquiry_df['category'].value_counts().to_frame('건수'))
print('[긴급도별 건수] — 도입부에서 물었던 그것')
display(inquiry_df['urgency'].value_counts().to_frame('건수'))
print('주문번호가 있는 문의:', inquiry_df['order_number'].notna().sum(), '건')

> `order_number` 를 보세요. **주문번호가 적힌 문의에만 값이 있고 나머지는 `None`** 입니다. `Optional` + "없으면 null" 지침이 없으면 모델은 빈칸을 싫어해서 **그럴듯한 번호를 지어냅니다** — 정형화에서 가장 위험한 실수입니다.

### 🖐️ 함께 따라하기 — 스키마에 필드를 하나 더해 보기

스키마를 **직접 고쳐 보는 것**이 이 단원의 핵심 연습입니다. "이 문의는 사람이 직접 봐야 하는가"를 판단하는 필드 하나를 더해, 6건 중 몇 건이 걸리는지 확인해 봅니다. **`description` 을 어떻게 쓰느냐**에 따라 결과가 달라지는 것도 함께 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) Inquiry 를 상속하지 말고, 위 Inquiry 정의를 복사해 필드 하나를 더한 InquiryV2 를 만든다
#    needs_human: bool = Field(description='...')  ← description 은 직접 쓴다
#    (예: '환불·불만처럼 금전이나 항의가 걸린 문의면 True, 단순 정보 문의면 False')
# 2) analyze_inquiry 와 같은 방식으로 부르되 response_format=InquiryV2 로 6건을 모두 분석한다
# 3) True 가 몇 건인지 세어 출력한다

### ✅ 바로 확인 퀴즈

**1.** 정형화한 문의 표에서 `order_number` 열이 **한 칸도 비어 있지 않았습니다.** 좋은 신호일까요, 나쁜 신호일까요? 무엇을 확인해야 할까요?

<details><summary>정답 보기</summary>

**나쁜 신호일 가능성이 큽니다.** 모든 문의에 주문번호가 적혀 있는 경우는 드물기 때문입니다 — 모델이 빈칸을 피하려고 **번호를 지어냈을** 수 있습니다. 원문 몇 건을 열어 실제로 번호가 있는지 대조하고, 필드가 `Optional`(기본값 `None`)인지, `description` 에 "없으면 null"이 적혀 있는지 확인합니다.

</details>

**2.** 어떤 팀이 `category: str` 로 문의 1,000건을 분류했더니 `value_counts()` 결과가 **237종류**로 나왔습니다. 무엇이 문제이고, 이미 만든 표는 어떻게 살릴 수 있을까요?

<details><summary>정답 보기</summary>

자유 문자열이라 '환불'·'환불요청'·'refund'·'환불 문의' 가 **다 다른 값**으로 흩어진 것입니다. 근본 해결은 `Literal` 로 후보를 못박고 **다시 돌리는 것**입니다. 다시 돌릴 여유가 없다면 237종류를 몇 개 대분류로 **매핑 표를 만들어 접는** 방법이 있지만, 그 매핑을 사람이 유지해야 하니 결국 비용은 비슷해집니다 — 스키마에서 막는 편이 쌉니다.

</details>

---
# 3. 개인정보 마스킹 — 중첩 스키마로 찾고 가리기

## 왜 필요할까요?
고객 문의·상담 기록에는 이름·전화번호·주소가 섞여 들어옵니다. 이 데이터를 분석하거나 외부에 넘기려면 **개인정보를 가려야** 합니다. 정규표현식으로 전화번호·이메일은 잡히지만, **이름과 주소**는 형식이 일정하지 않아 규칙으로 잡기 어렵습니다 — 여기가 LLM 이 필요한 자리입니다.

<img src="images/개인정보_탐지_마스킹.jpg" width="820">

*왼쪽: 문장 안에서 개인정보에 해당하는 조각만 찾아낸다(탐지). 오른쪽: 그 조각만 가리고 **나머지 내용은 그대로** 남긴다(마스킹).*

## 중첩 스키마
한 문장에 개인정보가 **여러 개** 나올 수 있으니, 항목마다 객체를 만들어 **리스트**로 받습니다. 앞 시간의 측면별 감성(ABSA)과 같은 모양입니다.

```
PIIResult
 ├─ masked_text : 개인정보를 가린 전체 문장
 ├─ items       : [ PIIItem, PIIItem, ... ]   ← 중첩
 │                  ├─ original : 원본 값
 │                  ├─ pii_type : 유형(이름·전화·이메일·주소·주민번호·카드·계좌)
 │                  └─ masked   : 가린 값
 └─ risk        : 전체 위험도
```

In [ ]:
class PIIItem(BaseModel):
    """문장에서 찾아낸 개인정보 한 건."""
    original: str = Field(description='탐지된 원본 값 그대로')
    pii_type: Literal['이름', '전화번호', '이메일', '주소', '주민번호', '카드번호', '계좌번호'] = Field(
        description='개인정보 유형')
    masked: str = Field(description='가린 값. 이름은 홍**, 전화는 010-****-5678, 이메일은 h***@gmail.com 형태')

class PIIResult(BaseModel):
    """문장 하나의 개인정보 탐지·마스킹 결과."""
    masked_text: str = Field(description='개인정보를 masked 값으로 바꾼 전체 문장')
    items: list[PIIItem] = Field(description='탐지된 개인정보 목록. 없으면 빈 리스트')
    risk: Literal['없음', '낮음', '보통', '높음'] = Field(
        description='주민번호·카드·계좌가 있으면 높음, 이름·전화·주소는 보통, 아무것도 없으면 없음')

print('중첩 스키마 준비 완료')

In [ ]:
# [제공 코드] 개인정보가 섞인 문장 4건 (모두 가상의 값입니다)
pii_samples = [
    '안녕하세요, 홍길동입니다. 010-1234-5678로 연락주세요. 이메일은 hong@example.com 입니다.',
    '주문자: 김영희, 전화: 02-9876-5432, 배송지: 부산시 해운대구 센텀로 99',
    '고객정보 - 성명: 박철수, 계좌: 국민은행 123-456-789012',
    '일반적인 상품 문의입니다. 재료와 사이즈에 대해 알고 싶어요.',
]
print(len(pii_samples), '건')

In [ ]:
def mask_pii(text):
    """문장에서 개인정보를 찾아 가린 결과를 돌려준다."""
    resp = client.chat.completions.parse(model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': '너는 개인정보보호 담당자야. 문장에서 개인정보를 찾아 스키마대로 가려라.'},
            {'role': 'user', 'content': text}],
        response_format=PIIResult)
    return resp.choices[0].message.parsed

for text in pii_samples:
    r = mask_pii(text)
    print('원문 :', text)
    print('가림 :', r.masked_text)
    print('탐지 :', [(i.pii_type, i.original, '→', i.masked) for i in r.items], '| 위험도:', r.risk)
    print()

## 가려졌는지 **확인**한다 — 말만 믿지 않기

`items` 에 "이건 개인정보다"라고 적어 놓고 **정작 `masked_text` 에는 원문이 그대로 남는** 일이 실제로 일어납니다. 적어 낸 것과 가린 것이 따로 노는 것이죠. 이건 코드로 바로 확인할 수 있습니다 — **탐지한 원본 값이 가린 문장에 남아 있으면 안 된다.**

In [ ]:
# 불변식: 탐지한 원본 값은 가린 문장에 남아 있으면 안 된다
leaks = []
for text in pii_samples:
    r = mask_pii(text)
    for it in r.items:
        if it.original in r.masked_text:
            leaks.append((it.pii_type, it.original))

print('가려졌다고 적어 놓고 실제로는 남아 있는 값:', len(leaks), '건')
for t_, v in leaks:
    print('  -', t_, ':', v)
print('\n→ 0 건이 아니면, 그 문장은 사람이 다시 봐야 합니다.')

> 마지막 문장처럼 **개인정보가 없으면 `items` 가 빈 리스트**여야 합니다. "찾아라"라고만 하면 모델이 억지로 뭔가를 찾아내므로, `description` 에 **"없으면 빈 리스트"** 를 적어 두는 것이 중요합니다.

**규칙 vs LLM** — 전화번호·이메일처럼 **형식이 고정된 것은 정규표현식이 더 싸고 정확**합니다. 이름·주소처럼 **형식이 없는 것**에만 LLM 을 쓰는 것이 실무의 균형점입니다. 둘을 섞어 쓰세요.

### 🖐️ 함께 따라하기 — 마스킹 규칙을 바꿔 보기

가리는 **정도**는 `description` 한 줄로 정해집니다. 지금은 전화번호 뒤 4자리를 남기는데, "전부 가린다"로 바꾸면 결과가 어떻게 달라지는지 직접 확인해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) PIIItem 을 복사해 masked 의 description 만 바꾼 PIIItemStrict 를 만든다
#    (예: '전부 가린다. 이름은 ***, 전화번호는 ***-****-****, 이메일은 ***@*** 형태')
# 2) 그 항목을 쓰는 PIIResultStrict 도 만든다(items 타입만 바꾸면 된다)
# 3) pii_samples[0] 을 두 스키마로 각각 분석해 masked_text 를 나란히 출력한다

### ✅ 바로 확인 퀴즈

**1.** 한 문장에서 개인정보가 여러 개 나올 수 있다는 것을 스키마로 어떻게 표현했나요?

<details><summary>정답 보기</summary>

**`items: list[PIIItem]`** — 객체를 **리스트로 중첩**했습니다. 측면별 감성에서 `aspects: list[AspectSentiment]` 를 쓴 것과 같은 모양입니다.

</details>

**2.** 상담 기록 10만 건을 마스킹해야 합니다. 전부 LLM 에 넣으면 비용이 큽니다. 어떻게 나눠 처리하면 좋을까요?

<details><summary>정답 보기</summary>

**형식이 고정된 것부터 규칙으로 걷어냅니다** — 전화번호·이메일·주민번호·카드번호는 정규표현식이 LLM 보다 싸고 정확합니다. 그렇게 걸러낸 뒤 **이름·주소처럼 규칙으로 못 잡는 것만** LLM 에 넘깁니다. 그러면 호출 건수가 크게 줄고, 규칙이 잡은 부분은 결과도 안정적입니다.

</details>

---
# 4. 설문 응답 표준화 — 제각각인 답을 같은 칸에

## 왜 필요할까요?
자유 입력 설문은 이렇게 들어옵니다.

```
"나이: 스물다섯살, 성별: 남자, 직업: 회사원, 연봉: 3000만원정도"
"25세 여성 개발자입니다. 연소득 3천5백. 경기도 거주"
"thirty years old, male, teacher, 2800만원, 부산시"
```

같은 항목인데 **표기가 전부 다릅니다**('스물다섯살'·'25세'·'thirty'). 평균 연령을 내려면 먼저 숫자로 만들어야 합니다. 그리고 **범위 표현('40대 초반')은 정확한 숫자가 없으므로 비워 두어야** 합니다 — 여기서도 `Optional` 이 핵심입니다.

## 두 층으로 받는다
- **원본 추출** — 응답에 실제로 적힌 값(`age`·`gender`·`job`·`income`·`region`)
- **표준화** — 분석용으로 카테고리화한 값(`age_group`·`job_category`·`income_range`)
- **품질 등급** — 이 응답을 얼마나 믿을 수 있는가(`quality`)

In [ ]:
class Standardized(BaseModel):
    """분석용으로 카테고리화한 값."""
    age_group: Optional[str] = Field(default=None, description="'20대','30대' 형태. 나이를 모르면 null")
    job_category: Optional[str] = Field(default=None,
        description="사무직·전문직·서비스직·자영업·학생·무직·기타 중 하나")
    income_range: Optional[str] = Field(default=None,
        description="'3000만원 미만','3000-5000만원','5000만원 이상' 중 하나. 모르면 null")

class Survey(BaseModel):
    """설문 응답 한 건을 검증·표준화한 결과."""
    age: Optional[int] = Field(default=None,
        description='정확한 나이 숫자만. 스물다섯살→25. 40대 초반처럼 범위면 반드시 null')
    gender: Literal['남성', '여성', '불명'] = Field(description='응답자 성별. 정보가 없으면 불명')
    job: Optional[str] = Field(default=None, description='응답에 적힌 직업 표현 그대로. 없으면 null')
    income: Optional[int] = Field(default=None, description='연소득을 만원 단위 숫자로. 3천5백→3500')
    region: Optional[str] = Field(default=None, description="시/도 수준만. '서울','경기','부산' 등")
    standardized: Standardized = Field(description='분석용 표준화 값')
    quality: Literal['우수', '양호', '보통', '불량'] = Field(
        description='우수: 모든 항목이 명확 / 양호: 일부 불분명 / 보통: 절반만 신뢰 / 불량: 대부분 불분명')

print('스키마 준비 완료 —', list(Survey.model_fields))

In [ ]:
# [제공 코드] 자유 입력 설문 응답 5건
survey_responses = [
    '나이: 스물다섯살, 성별: 남자, 직업: 회사원, 연봉: 3000만원정도, 지역: 서울',
    '25세 여성 개발자입니다. 연소득 3천5백. 경기도 거주',
    'thirty years old, male, teacher, 2800만원, 부산시',
    '나이 모름, 직업: 프리랜서, 돈: 많이벌어요 ㅋㅋ, 서울 강남 살아요',
    '40대 초반 남성 자영업자 연수입 5000만원 충청도',
]
print(len(survey_responses), '건')

In [ ]:
def standardize(text):
    """설문 응답 한 건을 Survey 스키마로 표준화한다."""
    resp = client.chat.completions.parse(model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': (
                '너는 설문 데이터 품질 관리자야. 응답을 스키마대로 추출·표준화해라.\n'
                '가장 중요한 규칙: 응답에 적혀 있지 않은 값은 절대 추측하지 말고 null 로 둔다.\n'
                "- 나이: '40대 초반'처럼 범위만 있으면 age 는 null, age_group 만 채운다.\n"
                "- 소득: '많이벌어요' 처럼 숫자가 없으면 income 과 income_range 를 둘 다 null.\n"
                "- 지역: 시/도 수준까지만 ('서울 강남' → '서울').\n"
                '- 품질: 나이·소득처럼 핵심 항목이 비어 있으면 우수/양호를 주지 말 것.'
            )},
            {'role': 'user', 'content': text}],
        response_format=Survey)
    return resp.choices[0].message.parsed

rows = []
for text in survey_responses:
    s = standardize(text)
    rows.append({'나이': s.age, '성별': s.gender, '연령대': s.standardized.age_group,
                 '직업분류': s.standardized.job_category, '소득구간': s.standardized.income_range,
                 '지역': s.region, '품질': s.quality})

survey_df = pd.DataFrame(rows)
display(survey_df)

> **4번 응답('나이 모름', '돈: 많이벌어요 ㅋㅋ')** 과 **5번 응답('40대 초반')** 을 보세요. 나이와 소득이 **비어 있으면(`None`) 제대로 동작한 것**입니다. 5번은 정확한 나이가 없으니 `나이` 는 비고 `연령대` 만 '40대' 로 채워집니다. **모르는 것을 비워 두는 것**이 지어내는 것보다 훨씬 낫습니다 — 빈칸은 나중에 채울 수 있지만, 지어낸 값은 **틀린 줄도 모르고** 분석에 섞여 들어갑니다.

## ⚠️ 그런데 `품질` 열을 보세요 — 지시가 안 먹힌 자리
시스템 프롬프트에 **"핵심 항목이 비어 있으면 우수/양호를 주지 말 것"** 이라고 적었는데도, 나이·소득이 통째로 빈 4번 응답까지 **'우수'** 로 나옵니다. 1교시에서 말한 그대로입니다 — **`system` 은 부탁이지 강제가 아닙니다.**

여기서 갈림길이 생깁니다.

| 맡길 일 | 누구에게 |
|---|---|
| "스물다섯살" → `25`, "3천5백" → `3500` 같은 **읽고 해석하기** | **모델** — 규칙으로 못 한다 |
| "빈 칸이 2개면 품질 보통" 같은 **셀 수 있는 판단** | **코드** — 모델에 맡길 이유가 없다 |

품질 등급은 두 번째입니다. 빈 칸을 세면 끝나는 일이니 **코드로 다시 계산**합시다.

In [ ]:
# 빈 칸 수로 품질을 직접 매긴다 — 세기만 하면 되는 판단은 코드가 더 정확하다
check_cols = ['나이', '연령대', '직업분류', '소득구간', '지역']
n_missing = survey_df[check_cols].isna().sum(axis=1)

def grade(n):
    return ['우수', '양호', '보통'][n] if n < 3 else '불량'

survey_df['빈칸수'] = n_missing
survey_df['품질(코드)'] = n_missing.map(grade)
display(survey_df[['나이', '소득구간', '빈칸수', '품질', '품질(코드)']])

diff = int((survey_df['품질'] != survey_df['품질(코드)']).sum())
print(f'모델 등급과 코드 등급이 다른 응답: {diff} / {len(survey_df)} 건')

> 두 등급이 갈리는 자리가 바로 **모델을 믿으면 안 되는 자리**입니다. 실무에서는 이렇게 **모델의 판단과 코드의 판단을 나란히 두고 어긋나는 행만 사람이 확인**합니다. 모든 행을 사람이 보는 것보다 훨씬 싸고, 모델만 믿는 것보다 훨씬 안전합니다.

In [ ]:
print('평균 나이(값이 있는 응답만):', survey_df['나이'].mean())
print('\n[품질 등급 분포 — 코드로 매긴 것]')
display(survey_df['품질(코드)'].value_counts().to_frame('건수'))
print('분석에 쓸 수 있는 응답(우수·양호):',
      int(survey_df['품질(코드)'].isin(['우수', '양호']).sum()), '/', len(survey_df))

### 🖐️ 함께 따라하기 — 자유 문자열을 `Literal` 로 바꿔 보기

`job_category` 의 `description` 에는 "사무직·전문직·서비스직·자영업·학생·무직·기타 중 하나"라고 적었지만, 타입이 `Optional[str]` 이라 위 결과에는 **'회사원'·'교육'·'IT·개발'** 처럼 제각각 들어왔습니다. `description` 은 부탁이고 **타입이 강제**입니다 — 직접 바꿔 확인해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) Standardized 를 복사해 job_category 만 Literal 로 바꾼 StandardizedV2 를 만든다
#    Literal['사무직','전문직','서비스직','자영업','학생','무직','기타']
# 2) 그것을 쓰는 SurveyV2 도 만든다(standardized 타입만 바꾸면 된다)
# 3) survey_responses 5건을 다시 분석해 job_category 값들을 출력하고, 위 표와 비교한다

### ✅ 바로 확인 퀴즈

**1.** 설문 200건을 정형화해 **평균 연령 34.2세**를 보고했습니다. 그런데 원본에는 '40대 초반'·'서른쯤'처럼 범위로 답한 사람이 60명 있었습니다. 이 평균을 그대로 써도 될까요?

<details><summary>정답 보기</summary>

**쓰면 안 됩니다.** 그 60명의 나이가 `None` 이었다면 평균은 나머지 140명만의 값이므로 "200명의 평균"이라고 말할 수 없고, 반대로 모델이 45·30 같은 숫자로 **채워 넣었다면** 지어낸 값이 평균에 섞인 것입니다. 어느 쪽인지 먼저 확인하고, 보고할 때 **몇 명 기준인지**를 함께 적어야 합니다.

</details>

**2.** 시스템 프롬프트로 "핵심 항목이 비면 우수를 주지 마라"고 지시했는데도 품질이 '우수'로 나왔습니다. 이런 판단은 어떻게 처리하는 게 좋을까요?

<details><summary>정답 보기</summary>

**코드로 직접 계산**합니다. `system` 은 강제가 아니라 부탁이라, 종합 판단은 자주 어깁니다. '빈 칸이 몇 개인가'처럼 **세기만 하면 되는 일**은 모델에 맡길 이유가 없습니다. 모델에는 "스물다섯살 → 25" 같은 **해석**을 맡기고, 그 결과로 하는 **판정**은 코드가 합니다.

</details>

**3.** 원본 값(`age`·`job`)과 표준화 값(`age_group`·`job_category`)을 굳이 **둘 다** 받는 이유는?

<details><summary>정답 보기</summary>

표준화 값은 **집계용**이고, 원본 값은 **검증용**입니다. 나중에 분류가 이상해 보일 때 원본을 보면 모델이 잘못 읽은 것인지 응답이 원래 애매한 것인지 가릴 수 있습니다.

</details>

---
# 5. 대량 레이블링 파이프라인 — 실제 데이터 40건을 표로

## 왜 필요할까요?
지금까지는 5~6건짜리 예시였습니다. 실무에서는 수백~수만 건을 돌립니다. 그때 필요한 것은 세 가지입니다.

| 필요한 것 | 왜 |
|---|---|
| **진행 상황 표시** | 몇 분씩 걸리는데 화면이 멈춰 있으면 고장인지 알 수 없다 |
| **실패해도 계속** | 한 건이 실패했다고 전체가 멈추면 앞의 결과까지 날아간다 |
| **결과 저장** | 같은 데이터를 다시 돌리면 시간과 돈을 또 쓴다 |

앞서 쓴 자세밴드 리뷰 40건을 **감정 + 측면 + 개선요청 여부**로 레이블링해 CSV 로 남겨 봅니다.

In [ ]:
reviews = pd.read_csv('data/reviews.csv')
print('리뷰:', reviews.shape)
display(reviews.head(3))

In [ ]:
class ReviewLabel(BaseModel):
    """리뷰 한 건의 레이블."""
    sentiment: Literal['긍정', '부정', '중립'] = Field(description='리뷰 전체 감정')
    topic: Literal['착용감', '효과', '내구성', '가격', '배송', '기타'] = Field(
        description='리뷰가 주로 말하는 주제')
    has_request: bool = Field(description='개선·교환·환불 요청이 담겨 있으면 True')

def label_review(text):
    resp = client.chat.completions.parse(model='gpt-4o-mini',
        messages=[{'role': 'system', 'content': '리뷰를 스키마대로 레이블링해라.'},
                  {'role': 'user', 'content': str(text)}],
        response_format=ReviewLabel)
    return resp.choices[0].message.parsed

In [ ]:
# 40건 일괄 처리 — 진행 상황을 찍고, 한 건이 실패해도 멈추지 않는다
labels = []
for i, (_, row) in enumerate(reviews.iterrows(), start=1):
    try:
        r = label_review(row['content'])
        labels.append({'review_id': row['review_id'], 'rating': row['rating'],
                       'sentiment': r.sentiment, 'topic': r.topic, 'has_request': r.has_request})
    except Exception as e:                     # 한 건 실패가 전체를 멈추지 않게
        print(f'  [{i}] 실패:', type(e).__name__)
        labels.append({'review_id': row['review_id'], 'rating': row['rating'],
                       'sentiment': None, 'topic': None, 'has_request': None})
    if i % 10 == 0:
        print(f'  {i}/{len(reviews)} 건 완료')

label_df = pd.DataFrame(labels)
print('\n완료:', len(label_df), '건 | 실패:', int(label_df['sentiment'].isna().sum()), '건')
display(label_df.head())

표가 되었으니 **별점과 대조**해 볼 수 있습니다. 별점은 사람이 매긴 값, `sentiment` 는 모델이 매긴 값이므로 **두 값이 얼마나 맞는지**가 곧 레이블링 품질입니다.

In [ ]:
# 사람이 준 별점 vs 모델이 준 감정 — 교차표로 한눈에
cross = pd.crosstab(label_df['rating'], label_df['sentiment'])
display(cross)

print('[주제별 건수]')
display(label_df['topic'].value_counts().to_frame('건수'))
print('개선·교환 요청이 담긴 리뷰:', int(label_df['has_request'].sum()), '건')

In [ ]:
# 결과를 저장한다 — 다시 돌리지 않아도 되도록
import os
os.makedirs('output', exist_ok=True)
out_path = 'output/review_labels.csv'
label_df.to_csv(out_path, index=False, encoding='utf-8-sig')
print('저장 완료:', out_path)
print(pd.read_csv(out_path).shape, '로 다시 읽힙니다')

> `encoding='utf-8-sig'` 는 엑셀에서 한글이 깨지지 않게 하는 표시입니다(11일차에서 배운 그것). **LLM 호출 결과는 반드시 저장하세요** — 같은 데이터를 다시 돌리면 시간도 돈도 다시 듭니다.

### 🖐️ 함께 따라하기 — `try/except` 가 못 잡는 실패를 직접 잡기

`try/except` 는 **호출이 터질 때**를 대비합니다(네트워크 끊김·요금 한도 초과·잘못된 모델 이름 등). 그런데 더 무서운 것은 **호출이 성공했는데 답이 엉뚱한** 경우입니다 — 예외가 없으니 조용히 지나갑니다. 빈 문자열을 넣어 그 장면을 만들고, **결과를 검사하는 코드**를 직접 붙여 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) texts = [reviews.loc[0, 'content'], '', '   '] 를 만든다(정상 리뷰 1건 + 빈 문자열 + 공백)
# 2) 각각을 label_review 로 처리하되 try/except 로 감싼다 — 예외가 나는지 보라
# 3) 예외가 안 난다면, 결과를 그대로 믿지 말고 **입력을 먼저 검사**하는 줄을 추가한다
#    (내용이 없으면 호출하지 말고 None 을 남긴다)

### ✅ 바로 확인 퀴즈

**1.** 빈 문자열을 넣었더니 예외가 아니라 **엉뚱한 레이블**이 돌아왔습니다. 이런 실패는 `try/except` 로 잡히지 않습니다. 그럼 무엇으로 잡아야 할까요?

<details><summary>정답 보기</summary>

**결과를 검사하는 코드**로 잡습니다. 호출이 성공했다고 답이 맞는 것은 아닙니다 — 입력이 비었는지 먼저 걸러 내거나(`if not text.strip(): continue`), 별점처럼 **이미 아는 값과 대조**해 이상한 행을 찾아냅니다. `try/except` 는 **터지는 실패**를, 검사는 **조용한 실패**를 담당합니다.

</details>

**2.** 리뷰 5,000건을 도는 파이프라인이 **3,200번째에서 네트워크 오류로 멈췄습니다.** `try/except` 가 없었다면 무엇을 잃었고, 있었다면 다음에 무엇만 다시 돌리면 될까요?

<details><summary>정답 보기</summary>

없었다면 **앞의 3,199건 결과까지 통째로** 잃습니다(메모리에만 있었으니까요). 있었다면 그 한 건만 `None` 으로 남고 나머지는 끝까지 진행되므로, 다음에는 **`sentiment` 가 비어 있는 행만** 골라 다시 돌리면 됩니다. 중간중간 CSV 로 저장해 두면 프로그램이 죽어도 같은 방식으로 이어서 할 수 있습니다.

</details>

**3.** 모델이 매긴 `sentiment` 가 맞는지 확인할 때, 이 데이터에서 무엇과 대조했나요?

<details><summary>정답 보기</summary>

**사람이 매긴 `rating`(별점)** 과 교차표로 대조했습니다. 1~2점인데 '긍정'이 많이 나온다면 레이블링이 잘못된 것입니다. 이렇게 **이미 있는 정답 비슷한 값**과 맞춰 보는 것이 품질 점검의 첫걸음입니다.

</details>

---
## 이번 강의 정리

| 주제 | 핵심 | 코드 |
|---|---|---|
| 정형화 | 자유 텍스트 → 같은 칸을 가진 표 | `parse(response_format=스키마)` |
| 없는 값 | 억지로 채우지 말고 비운다 | `Optional[str] = None` + "없으면 null" |
| 중첩 | 한 문장에 여러 건 | `items: list[PIIItem]` |
| 표준화 | 원본 + 카테고리 두 층 | `standardized: Standardized` |
| 대량 처리 | 진행 표시 · 실패 격리 · 저장 | `try/except`, `to_csv(...)` |

- **스키마 설계가 곧 프롬프트 설계**입니다. `description` 에 쓴 지침이 결과 품질을 좌우합니다.
- 규칙(정규표현식)으로 되는 것은 규칙으로, **판단이 필요한 것만** LLM 으로 — 이 균형이 비용을 좌우합니다.

## ⏭️ 예고 — 다음 시간

다음 시간에는 같은 일을 **이미지**로 합니다. 사진을 모델에 넣어 설명을 받고, 패션 사진과 영수증을 **표로 정형화**해 봅니다. 수고하셨습니다!